In [1]:
import os
import re
import csv
import pandas as pd
from pathlib import Path
from lxml import etree

In [63]:
DATA_DIR          = "../data"            
LOOKUP_CSV        = "../church-fathers.csv" 
OUTPUT_CSV        = "../sources_catalogue.csv"  
LOOKUP_SEP        = ";"   

LOOKUP_KEY_COL    = "CF_ID"               
LOOKUP_NAME_COL   = "Name"     

In [64]:

lookup_path = Path(LOOKUP_CSV)
lookup_df = pd.read_csv(lookup_path, sep=LOOKUP_SEP, dtype=str)
lookup_df.columns = lookup_df.columns.str.strip()

# Normalise the key column 
lookup_df[LOOKUP_KEY_COL] = lookup_df[LOOKUP_KEY_COL].str.strip().str.lower()

lookup = lookup_df.set_index(LOOKUP_KEY_COL).to_dict(orient="index")

print(f"Loaded {len(lookup)} entries from '{LOOKUP_CSV}'")
print(lookup_df.head())

Loaded 53 entries from '../church-fathers.csv'
   CF_ID bullinger_ID                     Name     gnd_id cc_id   class  \
0  cf001       P18881  Ignatius von Antiochien  118555340   NaN  syrian   
1  cf002       P19930      Polycarpus Smyrnäus  11859558X   NaN   greek   
2  cf003       P19894  Athenagoras Atheniensis  118646141   NaN   greek   
3  cf004       P21365   Theophilus Antiochenus  118756923   NaN   greek   
4  cf005       P18056         Irenäus von Lyon  118555766   NaN   greek   

    ogl_id pta_id Unnamed: 8  
0  tlg1443    NaN        NaN  
1  tlg1622    NaN        NaN  
2  tlg1205    NaN        NaN  
3  tlg1725    NaN        NaN  
4  tlg1447    NaN        NaN  


In [65]:
TEI_NS  = "http://www.tei-c.org/ns/1.0"
NS_MAP  = {"tei": TEI_NS}

def _text(element):
    """Return stripped text from an lxml element, or empty string."""
    if element is None:
        return ""
    return " ".join((element.text or "").split())


def extract_tei_metadata(xml_path: Path) -> dict:
    """
    Parse a TEI XML file and extract:
      - title   : //teiHeader/fileDesc/titleStmt/title
      - editor  : //teiHeader/fileDesc/titleStmt/editor  (first occurrence)
      - language: //teiHeader/profileDesc/langUsage/language/@ident
                  fallback → //text/@xml:lang
                  fallback → //TEI/@xml:lang
    """
    result = {"title": "", "editor": "", "language": ""}

    try:
        tree = etree.parse(str(xml_path))
        root = tree.getroot()

        # Detect whether TEI namespace is used
        ns = TEI_NS if root.tag.startswith("{") else ""
        pfx = "tei:" if ns else ""
        nm  = NS_MAP if ns else {}

        def xp(xpath_expr):
            """Run an XPath, auto-inserting the tei: prefix."""
            expr = xpath_expr.replace("tei:", pfx)
            return root.xpath(expr, namespaces=nm)

        # ── Title ────────────────────────────────────────────────────────────
        # Try the most specific path first, then progressively loosen
        for title_xpath in [
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title[@type='main']",
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            ".//tei:titleStmt/tei:title",
            ".//tei:title",
        ]:
            hits = xp(title_xpath)
            if hits:
                result["title"] = _text(hits[0])
                break

        # ── Editor ───────────────────────────────────────────────────────────
        for editor_xpath in [
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:editor",
            ".//tei:titleStmt/tei:editor",
            ".//tei:editor",
        ]:
            hits = xp(editor_xpath)
            if hits:
                # Collect all editors, join with '; '
                editors = [_text(e) for e in hits if _text(e)]
                if editors:
                    result["editor"] = "; ".join(editors)
                    break

        # ── Language ─────────────────────────────────────────────────────────
        # 1) langUsage/language @ident
        lang_nodes = xp(".//tei:profileDesc/tei:langUsage/tei:language")
        if lang_nodes:
            idents = [n.get("ident") or n.get("xml:lang") or n.get("{http://www.w3.org/XML/1998/namespace}lang") or _text(n)
                      for n in lang_nodes]
            idents = [i for i in idents if i]
            if idents:
                result["language"] = "; ".join(idents)

        # 2) fallback: @xml:lang on <text> element
        if not result["language"]:
            XML_NS = "http://www.w3.org/XML/1998/namespace"
            text_nodes = xp(".//tei:text") or xp(".//tei:TEI") or [root]
            for node in text_nodes:
                lang = node.get(f"{{{XML_NS}}}lang") or node.get("xml:lang")
                if lang:
                    result["language"] = lang
                    break

    except etree.XMLSyntaxError as exc:
        print(f"  ⚠ XML parse error in {xml_path.name}: {exc}")

    return result

In [66]:
data_root = Path(DATA_DIR)

records = []
unmatched_folders = set()
parse_errors = []

xml_files = sorted(data_root.rglob("*.xml"))
print(f"Found {len(xml_files)} XML files under '{DATA_DIR}/'\n")

for xml_path in xml_files:
    # Derive the subfolder name (immediate child of DATA_DIR)
    # Works even for deeply nested files — uses the first subfolder level
    rel = xml_path.relative_to(data_root)
    folder_key = rel.parts[0].strip().lower() if len(rel.parts) > 1 else "__root__"

    # Lookup church father csv
    entry = lookup.get(folder_key)

    if entry is None:
        unmatched_folders.add(folder_key)
        church_father = f"[unknown: {rel.parts[0]}]"
        cf_id = None
        bullinger_id = None
    else:
        church_father = entry.get(LOOKUP_NAME_COL)
        cf_id = folder_key
        bullinger_id = entry.get("bullinger_ID")

    # Extract TEI metadata
    meta = extract_tei_metadata(xml_path)

    records.append({
        "church_father": church_father,
        "CF_ID":    cf_id,
        "Bullinger_ID": bullinger_id,
        "work":          meta["title"],
        "language":      meta["language"],
        "source":        xml_path.name,          
        "editor":       meta["editor"],
    })

print(f"Processed {len(records)} files.")

if unmatched_folders:
    print(f"\n⚠ {len(unmatched_folders)} folder key(s) not found in lookup CSV:")
    for f in sorted(unmatched_folders):
        print(f"   • {f}")
    print("  → Add these keys to church-fathers.csv to resolve.")

Found 700 XML files under '../data/'

  ⚠ XML parse error in Chrysostom_In_Joannem.xml: Extra content at the end of the document, line 5, column 1 (Chrysostom_In_Joannem.xml, line 5)
Processed 700 files.


In [67]:
df = pd.DataFrame(records)

# output columns 
output_cols = ["church_father", "CF_ID", "Bullinger_ID", "work", "language", "source", "editor"]

df_out = df[output_cols].copy()

print(f"Shape: {df_out.shape}")
print(f"\nMissing values:")
print(df_out.isnull().sum())
print(f"\nEmpty strings:")
print((df_out == "").sum())

df_out.head(10)

Shape: (700, 7)

Missing values:
church_father     0
CF_ID             0
Bullinger_ID     64
work              0
language          0
source            0
editor            0
dtype: int64

Empty strings:
church_father      0
CF_ID              0
Bullinger_ID       0
work               2
language          17
source             0
editor           142
dtype: int64


,church_father,CF_ID,Bullinger_ID,work,language,source,editor
0,Ignatius von Antiochien,cf001,P18881,Epistulae vii genuinae,grc; lat,tlg1443.tlg001.1st1K-grc1.xml,Kirsopp Lake
1,Ignatius von Antiochien,cf001,P18881,Epistulae interpolatae et epistulae suppositic...,grc; lat,tlg1443.tlg002.1st1K-grc1.xml,Francis Xavier Funk; Francis Diekamp
2,Polycarpus Smyrnäus,cf002,P19930,Epistula ad Philippenses,grc; lat,tlg1622.tlg001.1st1K-grc1.xml,Kirsopp Lake
3,Athenagoras Atheniensis,cf003,P19894,Legatio sive Supplicatio pro Christianis,grc; lat,tlg1205.tlg001.perseus-grc1.xml,Johann Karl Theodor von Otto
4,Athenagoras Atheniensis,cf003,P19894,Supplication pro Christianis,grc; lat,tlg1205.tlg001.perseus-grc2.xml,Francis Andrew March; William Baxter Owen
5,Athenagoras Atheniensis,cf003,P19894,De Resurrectione,grc; lat,tlg1205.tlg002.perseus-grc1.xml,Johann Karl Theodor von Otto
6,Theophilus Antiochenus,cf004,P21365,Ad Autolycum,grc; lat,tlg1725.tlg001.perseus-grc1.xml,William Gilson Humphry
7,Irenäus von Lyon,cf005,P18056,Libros quinque adversus haereses,grc; lat,tlg1447.tlg001.1st1K-grc1.xml,William Wigan Harvey
8,Irenäus von Lyon,cf005,P18056,Fragmenta Synodicae Epistulae,grc; lat,tlg1447.tlg009.1st1K-grc1.xml,Martin Joseph Routh
9,Quintus Septimius Florens Tertullian,cf006,P17746,Responsa ex libris Digestorum,lat,001_Auctor-incertus_Responsa-ex-libris-Digesto...,Jacques-Paul Migne


In [68]:
# stats
print(f"Total files      : {len(df_out)}")
print(f"Church fathers   : {df_out['church_father'].nunique()}")
print()
print("Files per church father:")
print(df_out['church_father'].value_counts().to_string())
print()
print("Language distribution:")
print(df_out['language'].value_counts().to_string())

Total files      : 700
Church fathers   : 52

Files per church father:
church_father
Augustinus von Hippo                               155
Sophronius Eusebius Hieronymus                     104
Ambrosius von Mailand                               49
Quintus Septimius Florens Tertullian                37
Anicius Manlius Severinus Boethius                  32
Papst Gregor I. (der Große)                         26
Isidor von Sevilla                                  26
Prosper von Aquitanien                              25
Cyprian                                             24
Eusebius von Caesarea                               24
Hilarius von Poitiers                               23
Lucius Caecilius Firmianus Lactantius (Laktanz)     14
Claudius Gordianus Fulgentius                       14
Leo I. (der Große)                                  13
Gaius Marius Victorinus                             10
Rufinus von Aquileia                                10
Origenes                           

In [69]:
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  

In [5]:
from pathlib import Path
from lxml import etree
from collections import Counter
import json

TEI_NS = "http://www.tei-c.org/ns/1.0"

def survey_file(xml_path: Path) -> list[tuple]:
    """Return (depth, tag, type, subtype) tuples for all div-like elements in one file."""
    try:
        tree = etree.parse(str(xml_path))
    except etree.XMLSyntaxError:
        return []
    
    root = tree.getroot()
    body = root.find(f".//{{{TEI_NS}}}body")
    if body is None:
        body = root.find(".//body")
    if body is None:
        return []

    results = []
    for el in body.iter():
        if not isinstance(el.tag, str):  # skip comments, PIs
            continue
        tag = el.tag.replace(f"{{{TEI_NS}}}", "")
        if tag not in ("div", "div1", "div2", "div3"):
                continue
        
        # Depth relative to body
        depth = sum(1 for _ in el.iterancestors()) - sum(1 for _ in body.iterancestors()) - 1
        type_attr    = el.get("type", "")
        subtype_attr = el.get("subtype", "")
        n_attr       = el.get("n", "")
        
        results.append((depth, tag, type_attr, subtype_attr, n_attr != ""))

    return results


def main():
    sources_dir = Path("../data/")
    xml_files = sorted(sources_dir.rglob("*.xml"))
    print(f"Surveying {len(xml_files)} files...\n")

    # Count (depth, tag, type, subtype) combinations
    combo_counter  = Counter()
    # Track which files have which top-level div pattern
    top_level_patterns = Counter()

    for xml_path in xml_files:
        tuples = survey_file(xml_path)
        for t in tuples:
            combo_counter[t] += 1
        
        # Summarise top-level div pattern for this file
        top = tuple(sorted({(t[1], t[2], t[3]) for t in tuples if t[0] == 0}))
        top_level_patterns[top] += 1

    print("=== DIV COMBINATIONS (depth, tag, type, subtype, has_n) ===")
    print(f"{'depth':>6}  {'tag':<6}  {'type':<20}  {'subtype':<20}  {'has_n':<6}  count")
    print("-" * 75)
    for (depth, tag, typ, sub, has_n), count in sorted(combo_counter.items()):
        print(f"{depth:>6}  {tag:<6}  {typ:<20}  {sub:<20}  {str(has_n):<6}  {count}")

    print("\n=== TOP-LEVEL DIV PATTERNS PER FILE ===")
    for pattern, count in top_level_patterns.most_common():
        print(f"  {count:>4} files:  {pattern}")


if __name__ == "__main__":
    main()

Surveying 700 files...

=== DIV COMBINATIONS (depth, tag, type, subtype, has_n) ===
 depth  tag     type                  subtype               has_n   count
---------------------------------------------------------------------------
     0  div                                                 False   201
     0  div     edition                                     True    87
     0  div     praefatio                                   False   11
     0  div     translation                                 True    1
     0  div1                                                False   6652
     0  div1                                                True    79
     0  div1                          MSS                   False   5
     0  div1                          book                  True    284
     0  div1                          chapter               True    562
     0  div1                          dialog                True    3
     0  div1                          poem            